In [1]:
import dspy

In [2]:
model_ = "smollm2:360m-instruct-q4_K_M"

In [3]:
lm = dspy.LM(
    # model="groq/llama-3.1-8b-instant",
    model=f"ollama_chat/{model_}",
    temperature=0.2,
)
dspy.configure(lm=lm)


In [ ]:
class QA(dspy.Signature):
    """Helpful support answer grounded in provided context."""

    context: str = dspy.InputField(desc="Relevant knowledge base passages")

    user_question: str = dspy.InputField(desc="User's support question")

    ai_answer: str = dspy.OutputField(
        desc=(
            "Concise bullet-point answer using ONLY the provided context. "
            "If the answer is not present, say: 'I don't know based on the provided information."
            "Keep it strutured"
        )
    )

In [5]:
model = dspy.ChainOfThought(QA)

In [30]:
context = """
Introduction to LangChain
Last Updated : 12 Dec, 2025
LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for integrating with other tools and end-to-end chains for common applications. It helps AI developers connect LLMs such as GPT-4 with external data and computation. This framework comes for both Python and JavaScript.

Key benefits include:

Modular Workflow: Simplifies chaining LLMs together for reusable and efficient workflows.
Prompt Management: Offers tools for effective prompt engineering and memory handling.
Ease of Integration: Streamlines the process of building LLM-powered applications.
Key Components of LangChain
Lets see various components of Langchain:

langchain
1. Chains: Chains define sequences of actions, where each step can involve querying an LLM, manipulating data or interacting with external tools. There are two types:

Simple Chains: A single LLM invocation.
Multi-step Chains: Multiple LLMs or actions combined, where each step can take the output from the previous step.
2. Prompt Management: LangChain facilitates managing and customizing prompts passed to the LLM. Developers can use PromptTemplates to define how inputs and outputs are formatted before being passed to the model. It also simplifies tasks like handling dynamic variables and prompt engineering, making it easier to control the LLM's behavior.

3. Agents: Agents are autonomous systems within LangChain that take actions based on input data. They can call external APIs or query databases dynamically, making decisions based on the situation. These agents leverage LLMs for decision-making, allowing them to respond intelligently to changing input.

4. Vector Database: LangChain integrates with a vector database which is used to store and search high-dimensional vector representations of data. This is important for performing similarity searches, where the LLM converts a query into a vector and compares it against the vectors in the database to retrieve relevant information.

Vector database plays a key role in tasks like document retrieval, knowledge base integration or context-based search providing the model with dynamic, real-time data to enhance responses.

5. Models: LangChain is model-agnostic meaning it can integrate with different LLMs such as OpenAI's GPT, Hugging Face models, DeepSeek R1 and more. This flexibility allows developers to choose the best model for their use case while benefiting from LangChain’s architecture.

6. Memory Management: LangChain supports memory management allowing the LLM to "remember" context from previous interactions. This is especially useful for creating conversational agents that need context across multiple inputs. The memory allows the model to handle sequential conversations, keeping track of prior exchanges to ensure the system responds appropriately.
"""

In [ ]:
res = model(context=context, user_question="what is langchain")

In [43]:
model.inspect_history()





[2026-02-23T16:18:36.406967]

System message:

Your input fields are:
1. `context` (str): Relevant knowledge base passages
2. `user_question` (str): User's support question
Your output fields are:
1. `reasoning` (str): 
2. `ai_answer` (str): Concise bullet-point answer using ONLY the provided context. If the answer is not present, say: 'I don't know based on the provided information.Keep it strutured
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## user_question ## ]]
{user_question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## ai_answer ## ]]
{ai_answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Helpful support answer grounded in provided context.


User message:

[[ ## context ## ]]

Introduction to LangChain
Last Updated : 12 Dec, 2025
LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). 

In [33]:
from rich import print

In [34]:
print(res)

Prediction(
    reasoning='Introduction to LangChain',
    ai_answer='LangChain is an open-source framework designed to simplify the creation of applications using large 
language models (LLMs). It provides a standard interface for integrating with other tools and end-to-end chains for
common applications. It helps AI developers connect LLMs such as GPT-4 with external data and computation. This 
framework comes for both Python and JavaScript.'
)

In [40]:
print(res)

Prediction(
    reasoning='The user is asking for a general overview of LangChain. Based on the provided context, LangChain is 
an open-source framework designed to simplify the creation of applications using large language models (LLMs). It 
provides a standard interface for integrating with other tools and end-to-end chains for common applications.',
    ai_answer='Here are the key points about LangChain:\n- LangChain is an open-source framework for simplifying 
the creation of applications using large language models (LLMs).\n- It provides a standard interface for 
integrating with other tools and end-to-end chains for common applications.\n- It helps AI developers connect LLMs 
with external data and computation.\n- It supports both Python and JavaScript.'
)

In [ ]:
import dspy


class SupportQA(dspy.Signature):
    """Customer support answer grounded in knowledge base."""

    context: str = dspy.InputField(desc="Context from docs")
    question: str = dspy.InputField(desc="user question")

    answer: str = dspy.OutputField(
        desc=(
            "Concise bullet-point answer using ONLY the context. "
            "If not found, say: I don't know based on the provided information."
        )
    )

In [ ]:
# qa = dspy.Predict(SupportQA)
qa = dspy.ChainOfThought(SupportQA)


In [ ]:
FALLBACK = "i don't know based on the provided information"


def support_metric(example, pred, trace=True):
    ans = pred.answer.lower()
    ctx = example.context.lower()
    gold = example.answer.lower()

    if FALLBACK in ans:
        return FALLBACK in gold

    return any(sent.strip(" -•") in ctx for sent in ans.split("\n") if sent.strip())

In [137]:
trainset = [
    dspy.Example(
        context="Refunds are processed within 5 to 7 business days after approval.",
        question="How long does a refund take?",
        answer="Refunds are processed within 5 to 7 business days after approval.",
    ),
    dspy.Example(
        context="Two-factor authentication can be enabled from Security Settings.",
        question="How do I enable 2FA?",
        answer="Enable two-factor authentication from Security Settings.",
    ),
    dspy.Example(
        context="Password reset: click Forgot Password on login page.",
        question="How do I reset my password?",
        answer="Click Forgot Password on the login page.",
    ),
    dspy.Example(
        context="Support hours are 9 AM to 5 PM IST Monday to Friday.",
        question="Do you support weekends?",
        answer="I don't know based on the provided information.",
    ),
    # --- new 6 examples ---
    dspy.Example(
        context="You can update your email address from Account Settings under Profile.",
        question="How can I change my email address?",
        answer="Go to Account Settings and update your email under Profile.",
    ),
    dspy.Example(
        context="You can download invoices from the Billing section of your dashboard.",
        question="Where can I get my invoices?",
        answer="Download invoices from the Billing section of your dashboard.",
    ),
    dspy.Example(
        context="Our mobile app is available on iOS and Android.",
        question="Is there a Windows app?",
        answer="I don't know based on the provided information.",
    ),
    dspy.Example(
        context="To delete your account, contact support from the Help Center.",
        question="How do I delete my account?",
        answer="Contact support from the Help Center to delete your account.",
    ),
]

trainset = [ex.with_inputs("context", "question") for ex in trainset]

In [ ]:
from dspy.teleprompt import MIPROv2

optimizer = MIPROv2(
    metric=support_metric, auto="light", max_bootstrapped_demos=2, max_labeled_demos=2
)

optimized_qa = optimizer.compile(
    qa,
    trainset=trainset,
)

2026/02/23 17:56:28 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 6

2026/02/23 17:56:28 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/02/23 17:56:28 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/02/23 17:56:28 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 2/2 [00:00<00:00, 15.50it/s]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/6


100%|██████████| 2/2 [00:00<00:00, 35.95it/s]


...

2026/02/23 17:59:04 INFO dspy.evaluate.evaluate: Average Metric: 5 / 6 (83.3%)
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 83.33 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 4'].
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.33, 83.33, 66.67, 83.33, 83.33, 100.0, 66.67, 83.33, 83.33]
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 10 / 10 =====



Average Metric: 5.00 / 6 (83.3%): 100%|██████████| 6/6 [00:00<00:00, 98.22it/s] 

2026/02/23 17:59:04 INFO dspy.evaluate.evaluate: Average Metric: 5 / 6 (83.3%)
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 83.33 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 5'].
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.33, 83.33, 66.67, 83.33, 83.33, 100.0, 66.67, 83.33, 83.33, 83.33]
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====



Average Metric: 6.00 / 6 (100.0%): 100%|██████████| 6/6 [00:00<00:00, 226.65it/s]

2026/02/23 17:59:04 INFO dspy.evaluate.evaluate: Average Metric: 6 / 6 (100.0%)
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.33, 83.33, 66.67, 83.33, 83.33, 100.0, 66.67, 83.33, 83.33, 83.33, 100.0]
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 100.0
2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/02/23 17:59:04 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [140]:
optimized_qa.save("support_qa_44.json")

In [ ]:
res = optimized_qa(
    context="Password reset: click Forgot Password on login page.",
    question="How do I reset my password?",
)

print(res)

Prediction(
    reasoning='Not supplied for this particular example.',
    answer='Password reset: click Forgot Password on login page.'
)

In [ ]:
from rich import print

print(optimized_qa.inspect_history())





[2026-02-23T18:02:10.263611]

System message:

Your input fields are:
1. `context` (str): Context from docs
2. `question` (str): user question
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): Concise bullet-point answer using ONLY the context. If not found, say: I don't know based on the provided information.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Customer support answer grounded in knowledge base.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## context ## ]]
Refunds are processed within 5 to 7 business days after approval.

[[ ## question ## ]]
How long does a refund take?


Assistant message:

[[ ## reasoning ## ]]
Not supplied for this par

None

In [142]:
optimized_qa

predict = Predict(StringSignature(context, question -> reasoning, answer
    instructions='Customer support answer grounded in knowledge base.'
    context = Field(annotation=str required=True json_schema_extra={'desc': 'Context from docs', '__dspy_field_type': 'input', 'prefix': 'Context:'})
    question = Field(annotation=str required=True json_schema_extra={'desc': 'user question', '__dspy_field_type': 'input', 'prefix': 'Question:'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    answer = Field(annotation=str required=True json_schema_extra={'desc': "Concise bullet-point answer using ONLY the context. If not found, say: I don't know based on the provided information.", '__dspy_field_type': 'output', 'prefix': 'Answer:'})
))

In [151]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

In [153]:
# metric({'question':}, 'pred')